In [1]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = False
_AUGMENTED_PARAM = 'x_coef'
_AUGMENTED_EQUATION = 'OutGap'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.10
_MC_SAMPLES = 100_000
_MC_ALPHA = 0.05
_MC_SUMMARY_ONLY = True
_MC_INCLUDE_BY_PREDICTOR = False
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)



Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83  -0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [3]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [4]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [5]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
    summary_only=_MC_SUMMARY_ONLY,
    include_by_predictor=_MC_INCLUDE_BY_PREDICTOR,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [6]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: False
Augmented measurement equation: OutGap
Augmented coefficient: x_coef
Monte Carlo replications: 100000
Noise Covariance:
 [[1.208 0.    0.   ]
 [0.    1.679 0.   ]
 [0.    0.    0.078]]


In [7]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 100000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,19.056,0.001,0.023,0.0,100000,99635,0.996,0.0,0.996,0.997
1,Infl,30.503,0.000,0.027,0.0,100000,99999,1.000,0.0,1.000,1.000
2,Rate,23.476,0.000,0.025,0.0,100000,99938,0.999,0.0,0.999,1.000


In [8]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 100000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.123,2.201,0.615,0.000,0.007,0.001,100000,2566,0.026,0.001,0.025,0.027,3.0,200,4
1,cov_identity,19.373,202.818,0.000,0.007,0.164,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


In [9]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,-0.080,-0.004,1.384,-0.061,0.517,0.004,0.004,0.0,0.000,0.003,0.001,0.0,100000,3809,0.038,0.001,0.037,0.039
1,OutGap,x,-0.573,-0.237,0.166,-3.450,0.014,0.060,0.001,0.0,0.000,0.003,0.000,0.0,100000,93330,0.933,0.001,0.932,0.935
2,OutGap,r,-0.781,-0.028,1.943,-0.392,0.485,0.005,0.006,0.0,0.001,0.003,0.001,0.0,100000,5912,0.059,0.001,0.058,0.061
3,Infl,Pi,-1.182,-0.053,1.546,-0.755,0.418,0.008,0.005,0.0,0.000,0.003,0.001,0.0,100000,11513,0.115,0.001,0.113,0.117
4,Infl,x,0.103,0.038,0.191,0.544,0.455,0.007,0.001,0.0,0.000,0.003,0.001,0.0,100000,8546,0.085,0.001,0.084,0.087
5,Infl,r,-0.635,-0.020,2.175,-0.278,0.484,0.006,0.007,0.0,0.001,0.003,0.001,0.0,100000,6128,0.061,0.001,0.060,0.063
6,Rate,Pi,-0.287,-0.067,0.306,-0.949,0.379,0.009,0.001,0.0,0.000,0.003,0.001,0.0,100000,15111,0.151,0.001,0.149,0.153
7,Rate,x,0.026,0.048,0.038,0.687,0.434,0.007,0.000,0.0,0.000,0.003,0.001,0.0,100000,10087,0.101,0.001,0.099,0.103
8,Rate,r,-0.931,-0.151,0.426,-2.159,0.118,0.027,0.001,0.0,0.000,0.003,0.001,0.0,100000,56931,0.569,0.002,0.566,0.572


In [10]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-3.555,-0.279,0.863,-4.104,0.002,0.081,0.003,0.0,0.000,0.003,0.000,0.0,100000,99403,0.994,0.000,0.994,0.994
1,OutGap,x,-0.560,-0.365,0.100,-5.555,0.000,0.136,0.000,0.0,0.000,0.003,0.000,0.0,100000,99993,1.000,0.000,1.000,1.000
0,OutGap,r,1.549,0.059,1.841,0.832,0.434,0.006,0.005,0.0,0.001,0.002,0.001,0.0,100000,6541,0.065,0.001,0.064,0.067
5,Infl,Pi,-0.439,-0.030,1.005,-0.419,0.475,0.006,0.003,0.0,0.000,0.003,0.001,0.0,100000,6850,0.068,0.001,0.067,0.070
4,Infl,x,0.012,0.008,0.121,0.118,0.503,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,4795,0.048,0.001,0.047,0.049
3,Infl,r,-0.845,-0.028,2.062,-0.400,0.478,0.006,0.007,0.0,0.001,0.003,0.001,0.0,100000,6465,0.065,0.001,0.063,0.066
8,Rate,Pi,-0.108,-0.040,0.199,-0.564,0.460,0.006,0.001,0.0,0.000,0.003,0.001,0.0,100000,7720,0.077,0.001,0.076,0.079
7,Rate,x,0.012,0.033,0.024,0.473,0.474,0.006,0.000,0.0,0.000,0.003,0.001,0.0,100000,6733,0.067,0.001,0.066,0.069
6,Rate,r,-0.984,-0.168,0.403,-2.415,0.086,0.033,0.001,0.0,0.000,0.003,0.001,0.0,100000,66644,0.666,0.001,0.664,0.669


In [11]:
print("Innovation decomposition orthogonal summary:")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"])

Innovation decomposition orthogonal summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.464834,-1.544799,-0.079965,-0.079965,8.925261e-19,3.450555e-16,1.783286e-15,0.002690,0.002119,0.004135,0.004135,1.413995e-18,8.993017e-19,7.842785e-19
1,OutGap,x,0.043760,-0.616563,-0.572803,-0.572803,-2.322643e-19,7.534533e-17,1.783286e-15,0.000330,0.000304,0.000545,0.000545,3.268531e-19,2.237493e-19,7.842785e-19
2,OutGap,r,-0.325151,-0.455420,-0.780571,-0.780571,1.589637e-18,4.311036e-16,1.783286e-15,0.003833,0.003369,0.006015,0.006015,1.804234e-18,1.181848e-18,7.842785e-19
3,Infl,Pi,-0.044763,-1.137677,-1.182440,-1.182440,-1.663114e-17,3.897397e-16,1.783286e-15,0.001535,0.004733,0.004955,0.004955,1.632490e-18,1.071823e-18,7.842785e-19
4,Infl,x,0.004458,0.098723,0.103181,0.103181,-5.265284e-19,4.707098e-17,1.783286e-15,0.000190,0.000583,0.000611,0.000611,1.960774e-19,1.276406e-19,7.842785e-19
5,Infl,r,-0.026714,-0.608533,-0.635246,-0.635246,3.981131e-18,5.242822e-16,1.783286e-15,0.002170,0.006763,0.007052,0.007052,2.184184e-18,1.421996e-18,7.842785e-19
6,Rate,Pi,0.001522,-0.288058,-0.286536,-0.286536,3.048259e-19,1.391221e-16,1.783286e-15,0.000331,0.000903,0.000943,0.000943,5.591489e-19,3.451042e-19,7.842785e-19
7,Rate,x,-0.000301,0.026239,0.025938,0.025938,7.951052e-20,1.697143e-17,1.783286e-15,0.000041,0.000115,0.000119,0.000119,6.828638e-20,4.222287e-20,7.842785e-19
8,Rate,r,-0.012422,-0.918939,-0.931360,-0.931360,1.107314e-18,2.203484e-16,1.783286e-15,0.000465,0.001415,0.001451,0.001451,9.080821e-19,5.823113e-19,7.842785e-19


In [12]:
print("Innovation decomposition raw summary:")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"])

Innovation decomposition raw summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.785176,-5.340236,-3.555061,-3.555061,4.138356e-18,5.523820e-16,1.783286e-15,0.001747,0.001323,0.002601,0.002601,2.324516e-18,1.533706e-18,7.842785e-19
1,OutGap,x,0.183316,-0.743184,-0.559868,-0.559868,-4.234113e-19,7.388631e-17,1.783286e-15,0.000210,0.000185,0.000335,0.000335,3.198162e-19,2.183844e-19,7.842785e-19
2,OutGap,r,-0.669415,2.218911,1.549496,1.549496,-1.142420e-18,4.518175e-16,1.783286e-15,0.004248,0.003504,0.004503,0.004503,1.883425e-18,1.227147e-18,7.842785e-19
3,Infl,Pi,-0.010220,-0.428848,-0.439068,-0.439068,-1.862410e-17,2.458448e-16,1.783286e-15,0.001001,0.003050,0.003186,0.003186,1.018688e-18,6.609025e-19,7.842785e-19
4,Infl,x,0.000496,0.011404,0.011900,0.011900,-1.914834e-18,2.894568e-17,1.783286e-15,0.000120,0.000360,0.000376,0.000376,1.193571e-19,7.683871e-20,7.842785e-19
5,Infl,r,-0.030401,-0.814949,-0.845350,-0.845350,2.920198e-18,4.987777e-16,1.783286e-15,0.002051,0.006279,0.006522,0.006522,2.077575e-18,1.352253e-18,7.842785e-19
6,Rate,Pi,-0.000044,-0.108176,-0.108219,-0.108219,7.181430e-19,8.910467e-17,1.783286e-15,0.000216,0.000577,0.000602,0.000602,3.566339e-19,2.186223e-19,7.842785e-19
7,Rate,x,0.000001,0.011789,0.011790,0.011790,5.158094e-20,1.069371e-17,1.783286e-15,0.000026,0.000072,0.000074,0.000074,4.291603e-20,2.642435e-20,7.842785e-19
8,Rate,r,-0.008852,-0.975446,-0.984298,-0.984298,8.209278e-19,2.151076e-16,1.783286e-15,0.000440,0.001379,0.001414,0.001414,8.907193e-19,5.750393e-19,7.842785e-19


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.


In [13]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
        summary_only=_MC_SUMMARY_ONLY,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()

## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$

In [14]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,1.605,-2908.395,-1260.754,3295.283,0.0,0.0,0.735,0.129,1.389,0.0,100000,100000,1.0,0.0,1.0,1.0


In [15]:
res_mle

OptimizationResult(kind='mle', x=array([1.43867678]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(1.4386767775944067), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(1249.1509618619887), loglik=np.float64(-1249.1509618619887), logprior=np.float64(0.0), logpost=np.float64(-1249.1509618619887), nfev=18, nit=8, raw=  message: CONVERGENCE

## Serial Autocorrelation Tests for the Augmented Model

In [16]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,2.283,0.320,0.008,0.001,100000,20627,0.206,0.001,0.204,0.209
1,Infl,31.696,0.000,0.028,0.000,100000,99996,1.000,0.000,1.000,1.000
2,Rate,22.184,0.001,0.025,0.000,100000,99871,0.999,0.000,0.998,0.999


In [17]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.123,2.201,0.615,0.000,0.007,0.001,100000,2566,0.026,0.001,0.025,0.027,3.0,200,4
1,cov_identity,19.373,202.818,0.000,0.007,0.164,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.097,2.279,0.597,0.000,0.006,0.001,100000,2355,0.024,0.0,0.023,0.025,3.0,200,4
1,cov_identity,2.265,57.610,0.000,0.001,0.045,0.000,100000,100000,1.000,0.0,1.000,1.000,6.0,200,4


Reference-minus-augmented moment distance comparison:


""
